In [5]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [6]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [7]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [8]:
analize = Analizer(0.6)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_38_9_22,0.997579,0.794734,0.984012,0.981127,0.988194,0.016189,1.372617,0.020660,0.075975,0.048317,0.059641,0.127236,1.000796,0.132653,202.246853,320.477808,"Hidden Size=[24], regularizer=0.2, learning_ra..."
1,model_38_9_21,0.997578,0.794514,0.984374,0.981269,0.988321,0.016193,1.374089,0.020191,0.075402,0.047797,0.060955,0.127251,1.000796,0.132669,202.246364,320.477319,"Hidden Size=[24], regularizer=0.2, learning_ra..."
2,model_38_9_23,0.997577,0.794929,0.983682,0.980997,0.988078,0.016203,1.371309,0.021085,0.076499,0.048792,0.058461,0.127289,1.000797,0.132708,202.245165,320.476120,"Hidden Size=[24], regularizer=0.2, learning_ra..."
3,model_38_9_20,0.997574,0.794266,0.984773,0.981424,0.988461,0.016221,1.375747,0.019676,0.074777,0.047226,0.062417,0.127362,1.000798,0.132784,202.242900,320.473855,"Hidden Size=[24], regularizer=0.2, learning_ra..."
4,model_38_9_24,0.997573,0.795103,0.983383,0.980878,0.987973,0.016229,1.370146,0.021472,0.076976,0.049224,0.057403,0.127393,1.000798,0.132816,202.241926,320.472881,"Hidden Size=[24], regularizer=0.2, learning_ra..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3166,model_39_5_18,0.930597,0.798209,0.873747,0.943270,0.937807,0.464098,1.349376,0.306532,0.744993,0.525763,1.124174,0.681247,1.022817,0.710250,195.535319,313.766274,"Hidden Size=[24], regularizer=0.05, learning_r..."
3167,model_35_9_11,0.930585,0.700059,0.861236,0.888017,0.893499,0.464176,2.005704,1.246566,0.496042,0.871304,1.455289,0.681305,1.024144,0.710309,187.534983,300.890435,"Hidden Size=[23], regularizer=0.05, learning_r..."
3171,model_7_9_6,0.930548,0.736622,0.940528,0.804527,0.931030,0.464427,1.761209,0.103103,0.357900,0.230501,0.804873,0.681489,1.040655,0.710502,131.533900,210.760829,"Hidden Size=[16], regularizer=0.05, learning_r..."
3172,model_22_2_4,0.930547,0.761964,0.930092,0.855567,0.917496,0.464435,1.591746,0.574387,0.366425,0.470406,0.959092,0.681495,1.029244,0.710508,163.533865,262.262807,"Hidden Size=[20], regularizer=0.2, learning_ra..."
